# VaultGuard — Exploratory Data Analysis

This notebook explores `data/raw/creditcard.csv`; all figures are saved in `reports/figures`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}. Place creditcard.csv in data/raw/.')

sns.set_theme(style='whitegrid', context='notebook')
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape (rows, columns): {df.shape}')

## First look

Review the schema and descriptive statistics before modelling.

In [ ]:
display(df.head())
df.info()
display(df.describe().T)

## Data quality and class balance

The data is highly imbalanced: fraud is expected to be a very small share of transactions. Accuracy alone is not enough, because predicting every transaction as legitimate can still produce a high score. Evaluate later models with precision, recall, F1, ROC-AUC, and especially PR-AUC.

In [ ]:
print('Missing values by column:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print(f'Duplicate rows: {df.duplicated().sum():,}')

class_counts = df["isfraud"].value_counts().sort_index()
class_summary = pd.DataFrame({'count': class_counts, 'percentage': (class_counts / len(df) * 100).round(4)})
class_summary.index = class_summary.index.map({0: 'Legitimate', 1: 'Fraud'})
display(class_summary)
fraud_percentage = df["isfraud"].eq(1).mean() * 100
print(f'Fraud percentage: {fraud_percentage:.4f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='Class', hue='Class', palette=['#4C78A8', '#E45756'], legend=False, ax=ax)
ax.set(title='Legitimate vs Fraudulent Transactions', xlabel='Transaction class', ylabel='Number of transactions')
ax.set_xticks([0, 1], ['Legitimate', 'Fraud'])
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'class_count.png', dpi=300, bbox_inches='tight')
plt.show()

## Transaction amount and time

`Time` is elapsed seconds since the first transaction. The `V1`–`V28` fields are anonymized PCA-transformed variables and should not be assigned speculative real-world meanings.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=df, x='Amount', hue='Class', bins=100, element='step', stat='count', common_norm=False, ax=ax)
ax.set(title='Transaction Amount Distribution by Class', xlabel='Transaction amount', ylabel='Number of transactions')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'amount_distribution_by_class.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='Class', y='Amount', hue='Class', palette=['#4C78A8', '#E45756'], legend=False, ax=ax)
ax.set(title='Transaction Amount by Class', xlabel='Transaction class', ylabel='Transaction amount')
ax.set_xticks([0, 1], ['Legitimate', 'Fraud'])
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'amount_boxplot_by_class.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df.loc[df["isfraud"].eq(1)], x='Time', bins=75, color='#E45756', ax=ax)
ax.set(title='Fraud Activity Over Time', xlabel='Seconds elapsed since first transaction', ylabel='Number of fraudulent transactions')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fraud_activity_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

## EDA conclusion

The severe class imbalance is the central modelling concern. Use a stratified split and select model thresholds with fraud-sensitive metrics—not accuracy alone.